# 03/05 — Permutation null for attenuation slope

Shuffle treatment labels at the *mouse* level and recompute the regional attenuation slope. Empirical p per region vs the observed slope. With 3 vs 3 mice the permutation space is small (~20 unique permutations) so this is a sanity check, not a high-power test — but it falsifies obviously spurious slopes.

In [ ]:
from __future__ import annotations
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# regional-annotation h5ad lives on the processing volume
BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_ORIENTED = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
H5AD_BASE     = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD = H5AD_ORIENTED if H5AD_ORIENTED.exists() else H5AD_BASE
COUNT_LAYER = 'counts'   # raw integer counts live here, not in .X

TBL = ROOT / 'results' / 'tables' / 'attenuation'
TBL.mkdir(parents=True, exist_ok=True)
FIG = ROOT / 'results' / 'figures' / 'manuscript'
FIG.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY    = 'sample_id'           # change to 'library_id' if obs uses that
REGION_KEY    = 'anatomical_region'   # adjust to your obs column for regions
TREATMENT_KEY = 'treatment'
print('h5ad        :', H5AD)
print('count layer :', COUNT_LAYER)


In [ ]:
from itertools import permutations
from utils.attenuation import pseudobulk_lfc, attenuation_slope

counts = pd.read_csv(TBL / 'pseudobulk_counts.tsv',
                     sep='\t', index_col=0)
meta   = pd.read_csv(TBL / 'pseudobulk_meta.tsv',
                     sep='\t', index_col=0)

obs_stats = pd.read_csv(TBL / 'regional_attenuation_stats.tsv',
                        sep='\t', index_col=0)


In [ ]:
def slope_for(meta_local, region):
    pbs_v_wt = pseudobulk_lfc(counts, meta_local,
                              group_a='PBS', group_b='WT',
                              region=region)
    bri_v_wt = pseudobulk_lfc(counts, meta_local,
                              group_a='BRICHOS', group_b='WT',
                              region=region)
    common = pbs_v_wt.lfc.index.intersection(bri_v_wt.lfc.index)
    sig = pbs_v_wt.padj.loc[common] < 0.05
    return attenuation_slope(pbs_v_wt.lfc.loc[common],
                             bri_v_wt.lfc.loc[common],
                             sig, n_boot=0)['slope']

mice = meta[['sample', 'treatment']].drop_duplicates()
labels = mice['treatment'].values
samples = mice['sample'].values

perms = list(set(permutations(labels)))
print(f'{len(perms)} unique permutations')


In [ ]:
rng_perms = perms[:200]   # cap for speed
rows = []
for region in obs_stats.index:
    obs_slope = obs_stats.loc[region, 'slope']
    null_slopes = []
    for perm in rng_perms:
        local = meta.copy()
        mapping = dict(zip(samples, perm))
        local['treatment'] = local['sample'].map(mapping)
        try:
            null_slopes.append(slope_for(local, region))
        except Exception:
            continue
    null_slopes = np.array(null_slopes)
    p_left = (null_slopes <= obs_slope).mean()  # rescue test
    rows.append(dict(region=region, obs_slope=obs_slope,
                     null_median=np.nanmedian(null_slopes),
                     p_one_sided=p_left,
                     n_perms=len(null_slopes)))
perm_df = pd.DataFrame(rows).set_index('region')
perm_df.to_csv(TBL / 'permutation_null.tsv', sep='\t')
perm_df
